In [2]:
#Plotting
%matplotlib widget
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors
from matplotlib.colors import ListedColormap, BoundaryNorm, TwoSlopeNorm
import matplotlib.ticker as ticker
from matplotlib import patches
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D

#Mapping tools
import cartopy.crs as ccrs
import cartopy.io.img_tiles as cimgt
from cartopy.io import shapereader
from cartopy.feature import ShapelyFeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cmocean import cm


RuntimeError: 'widget is not a recognised GUI loop or backend name

In [6]:
%matplotlib widget
import matplotlib.pyplot as plt

plt.plot([1, 2, 3], [4, 5, 6])
plt.title("Interactive Widget Plot")
plt.show()

RuntimeError: 'widget is not a recognised GUI loop or backend name

In [ ]:
# Load the bathymetry data and convert it into an xarray dataset
ds2 = xr.open_dataset('Mapping/crm_socal_3as_vers2.nc')

dsub2 = ds2.sel(lon=slice(-122.45, -121.75), lat=slice(36.3, 37))

lat46042 = 36.785
lon46042 = -122.396

lat46239 = 36.335
lon46239 = -122.104

latSWC = 36.56196
lonSWC = -121.94176

SWCextent = ([-121.965,-121.925,36.551,36.570])

latM1 = 36.75
lonM1 =  -122

import matplotlib as mpl
# Plot the map projection with bathymetry contours and location of SWC001 mooring
plt.figure(figsize=(10,10))

font = {'family': 'Arial', 'size': 14, 'weight':'bold'}
mpl.rc('font', **font)
    
ax = plt.axes(projection=ccrs.Mercator())
gl = ax.gridlines(draw_labels=True)
gl.xlabels_top = False
gl.ylabels_right = False
gl.xformatter = LONGITUDE_FORMATTER
gl.yformatter = LATITUDE_FORMATTER
ax.set_extent([-122.45, -121.75, 36.3, 37])
cs = plt.contourf(dsub2.lon, dsub2.lat, dsub2.Band1, np.arange(-3000, 100, 100), transform=ccrs.PlateCarree(),cmap=cm.deep_r)
cbar = plt.colorbar(cs,location='top',shrink=.65, aspect=20, pad=.025)
cbar.set_label('Depth (m)',fontsize=14,fontname='Arial',fontweight='bold',va='bottom')
#cbar.ax.tick_params(labelsize=12)
    
# This is the fix for the white lines between contour levels
for c in cs.collections:
    c.set_edgecolor("face")
    
plt.contour(dsub2.lon, dsub2.lat, dsub2.Band1, [0], transform=ccrs.PlateCarree(), colors='k')

#Station 46042
plt.plot(lon46042, lat46042, 'ro',transform=ccrs.PlateCarree())
plt.text(lon46042, lat46042,' NDBC buoy \n 46042',color='k',va='center',ha='left',transform=ccrs.PlateCarree())

#MBARI M1
plt.plot(lonM1, latM1, 'ro',transform=ccrs.PlateCarree())
plt.text(lonM1, latM1,' MBARI M1 \n mooring',color='k',va='center',ha='left',transform=ccrs.PlateCarree())

#SWC Box
plt.plot([SWCextent[0],SWCextent[0],SWCextent[1],SWCextent[1],SWCextent[0]],
        [SWCextent[2],SWCextent[3],SWCextent[3],SWCextent[2],SWCextent[2]],
         'r-',linewidth=1.5,transform=ccrs.PlateCarree())
plt.text(-121.919, 36.553, 'Stillwater \n Cove',color='k',va='center',ha='left',transform=ccrs.PlateCarree())

#Station 46239
plt.plot(lon46239, lat46239, 'ro',transform=ccrs.PlateCarree())
plt.text(lon46239, lat46239,' NDBC buoy \n 46239',color='k',va='center',ha='left',transform=ccrs.PlateCarree())

gl.xlines = False
gl.ylines = False

gl.xlocator = ticker.FixedLocator(np.arange(-122.4,-121.7,0.2))
gl.ylocator = ticker.FixedLocator(np.arange(36.3,37,0.1))

#plt.savefig('Figures/MB_RegionalMap.png', dpi=600)

In [ ]:
filename_xyz = 'Maps/Mstr_CyPt2m_xyz.txt'
data = np.genfromtxt(filename_xyz)

x = data[:,0]
y = data[:,1]
z = data[:,2]

# find the extent and difference 
xmin=np.min(x);xmax=np.max(x)
ymin=np.min(y);ymax=np.max(y)
mdx=np.abs(np.diff(x))
mdy=np.abs(np.diff(y))

# determine dx and dy from the median of all the non-zero difference values
dx=np.median(mdx[np.where(mdx>0.0)[0]])
dy=np.median(mdy[np.where(mdy>0.0)[0]])

#construct x,y,z of complete grid
xi=np.arange(xmin,xmax+dx,dx)
yi=np.arange(ymin,ymax+dy,dy)
zi=np.ones((len(yi),len(xi)))*np.nan
np.shape(zi)

# calculate indices in full grid (zi) to stick the input z values
ix=np.round((x-xmin)/dx).astype(int)
iy=np.round((y-ymin)/dy).astype(int)
zi[iy,ix]=z

SWCextent = ([-121.96,-121.925,36.549,36.570])
SWCds = ds2.sel(lon=slice(-121.96,-121.925), lat=slice(36.549,36.570))

request = cimgt.GoogleTiles(style='satellite')
ms = 9

font = {'family': 'Arial', 'size': 14, 'weight':'bold'}
mpl.rc('font', **font)

plt.figure(figsize=(10,10))
ax,gl = make_map(projection=request.crs)
ax.set_extent(SWCextent)
ax.add_image(request,15)

cbar = plt.contour(xi,yi,zi,np.arange(0,300,10),transform=ccrs.UTM(10),linewidth=0.25,cmap='gist_earth_r');
plt.clabel(cbar, np.array((10.,30.,50.,100.,150.)), inline=True, colors='white', fontsize=14)
plt.clim([-50,250])

gl.xlines = False
gl.ylines = False
gl.xlocator = ticker.FixedLocator(np.arange(-121.96,-121.92,0.01))
gl.ylocator = ticker.FixedLocator(np.arange(36.55,36.570,0.005))

#Pescadero Point
plt.plot(-121.9522, 36.56132, 'ro', ms=6, transform=ccrs.PlateCarree())
plt.text(-121.9522, 36.56032,' Pescadero\n Point',color='w',va='center',ha='right',weight='bold',transform=ccrs.PlateCarree())

#Pescadero Rocks
plt.plot(-121.945157, 36.5625, 'ro', ms=6, transform=ccrs.PlateCarree())
plt.text(-121.9465, 36.5634,' Pescadero \n Rocks',color='w',va='center',ha='center',weight='bold',transform=ccrs.PlateCarree())

#Arrowhead Point
plt.plot(-121.9405, 36.560879, 'ro', ms=6, transform=ccrs.PlateCarree())
plt.text(-121.94045, 36.5608,' Arrowhead \n Point',color='w',va='center',ha='left',weight='bold',transform=ccrs.PlateCarree())

#SWC Mooring
plt.plot(lonSWC, latSWC, 'y*', ms=12, transform=ccrs.PlateCarree())
plt.text(lonSWC, latSWC+.0008,' Mooring',color='w',va='center',ha='center',weight='bold',transform=ccrs.PlateCarree())

#Carmel Bay
plt.text(-121.944, 36.551,' Carmel\n Bay',color='w',va='center',ha='center',weight='bold',transform=ccrs.PlateCarree())


#Scale Box
sc_lon = -121.935
sc_lat = 36.567

plt.plot([sc_lon,sc_lon+0.5/(111*np.sin(sc_lat*np.pi/180))],
         [sc_lat,sc_lat],
         'k-',linewidth=2,transform=ccrs.PlateCarree())
plt.text(sc_lon+0.25/(111*np.sin(sc_lat*np.pi/180)),sc_lat,'500 m',va='bottom',ha='center',
         color='k',fontname='Times New Roman',transform=ccrs.PlateCarree())

plt.gca().add_patch(patches.Rectangle((sc_lon-0.001,sc_lat-0.0012), 
                                      0.0095, 0.003,
                                      transform=ccrs.PlateCarree(),facecolor='w'))

#plt.title('B) Map of Stillwater Cove',loc='left',fontsize=14)
plt.show()

#plt.savefig('Figures/SWC_Map.png', dpi=600)